[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/closed_universe/hopf_projection.ipynb)

# Gravitational Waves Below the Hopf Circles

This example projects the gravitational waves of a closed universe, the round three-sphere, onto the ordinary sphere below its Hopf circles, and watches a quadrupole travel round it. The three-sphere is a bundle of great circles, its Hopf circles, each as far from every other all along it, and each circle stands over one point of an ordinary sphere, the base. A wave that is the same all along the circles loses nothing on the base: read along a circle it says how much the circle stretches, and read across the circles it is the strain of the base itself, which atoms on springs feel and light crossing the base bends to. Nothing is computed on the base: the waves, the atoms and the light all run in the three-sphere, lifted across the circles, and only what they read comes down to be drawn.

Points of the three-sphere are modelled as antivectors, and the circles as the orbits of a turn acting on the same side as the frames, `frame(place)`. Read in its frame, a circle's direction, `frame(place).inverse() >> place.commutator(circling)`, is the same step of the origin all along the circle: the base is the sphere of such steps, and that step is the circle's base point. Steps across the circles are the base's own directions turned a quarter in the plane square to the base point, `turning(base).commutator(direction)`, and the Hopf map doubles their angles. A strain read along a step against the round metric on the steps, `(strain(step) & step) / (2 * (polar(step) & step))`, is how much lengths stretch along it.

In quaternion notation, the Hopf map sends a unit quaternion $q$ to the unit imaginary quaternion $\bar q\,\mathbf{i}\,q$, the same all along the circle $e^{\mathbf{i}t} q$; the angles between those images are twice the angles across the circles, so that two circles a quarter turn apart map to antipodes. The 600-cell's vertices read there as the 120 unit icosians, a group of unit quaternions, and its decagons as the cosets of a cyclic subgroup of order ten.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np
from numga import NumpyContext, stack

from examples.util.animation import show_animation
from examples.util.topology import Topology
from examples.relativity.closed_universe import core, render, scenarios
from examples.relativity.closed_universe.core import ga

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                     # Scalar+
# Planes through the origin, the polar planes of its steps.
Plane = ga.gatype.vector()                                                      # Vector+
Through = Plane.derive.within("x y z")                                          # Vector+
# Points of the sphere and the origin's steps, the base's points among them; the motors that carry them.
Point = ga.gatype.antivector()                                                  # Antivector-
Step = Point.derive.containing("w")                                             # Antivector-
Motor = ga.gatype.rotor()                                                       # Rotor+
Translating = ga.gatype((Step, Through))                                        # Antivector- <- Vector+

# The origin's equator, the plane `w`, and the origin, its pole.
equator = mv.w                                                                  # [] Plane+
origin = equator.dual()                                                         # [] Point-
# The step the origin takes under the motion across a plane through it, a step by its value, cast so; its inverse
# gives each step its polar plane, the round sphere's metric on the origin's steps as a correlation, and the meet of
# the equator with that plane is the rate that moves the origin along the step.
polar = origin.commutator(equator ^ Through).cast(Translating).inverse()        # [] Plane+ <- Point-
moving = equator ^ polar(Step)                                                  # [] Rate+ <- Point-

## The frame at each place and the Hopf circles

In four dimensions the product of a rate with the extrinsic pseudoscalar `mv.xyzw` is the rate in the plane square to it, and as the pseudoscalar squares to one, it splits every rate into a half it keeps and a half it flips. Each half turns alike in two square planes, by the same angle in both, so that it moves every point of the sphere the same distance along a great circle. The quotient `place / origin` is the turn that carries the origin to a place in the plane of the great circle between them; its half that the pseudoscalar keeps, with one in the other half, still carries the origin there, turning alike in that plane and in the plane square to it: that motor is the frame at the place. A turn by a rate of that same half moves every point along a great circle of its own, and those circles, one through every point, are the Hopf circles. The pseudoscalar is what splits the sphere's turns into the two halves, and so what makes the circles: across them, `turning`, the moving rate times the pseudoscalar, turns a step a quarter in the plane square to the moving rate.

In [ ]:
# The extrinsic pseudoscalar, whose product with a rate is the rate in the plane square to it; the rate that turns a
# frame in the plane square to each step's moving rate.
pseudoscalar = mv.xyzw                                                          # [] Antiscalar+
turning = moving * pseudoscalar                                                 # [] Rate+ <- Point-


def frame(places: Point) -> Motor:
    """The frames at places on the sphere, `Motor+`: the motors that carry the origin to each by turning
    alike in the plane of the great circle between them and in the plane square to it."""
    # The half of the quotient of the place by the origin that the pseudoscalar keeps, with one in the other half.
    return ((1 + pseudoscalar) * (places.normalized() / origin) + (1 - pseudoscalar)) / 2

## The base below the circles

The 600-cell's guardrail decagons are Hopf circles: the orbits of the turn that carries each vertex a tenth of the way along its circle, a turn that carries the whole 600-cell onto itself, and their base points are the corners of an icosahedron. Since the Hopf map doubles angles, the circle above a base point lies half the angle across the circles from the origin's own.

In [ ]:
cell_mesh, cell_vertices = scenarios.six_hundred_cell()
turn = scenarios.circle_turn(cell_vertices, scenarios.n_guardrail_sides)        # [] Motor+
circling = turn.log()                                                           # [] Rate+


def base_points(places: Point) -> Point:
    """The base point below each place, `Point-`: the direction its circle runs in, read in the frame at
    the place, a unit step of the origin."""
    circle_directions = places.commutator(circling)                            # Point-
    return (frame(places).inverse() >> circle_directions).cast(Step).normalized()


def crossing(bases: Point, directions: Point) -> Point:
    """Steps along the base, `directions` at the base points, turned a quarter in the plane square to the base point,
    `Point-`: the steps across the circles that a base direction is the image of."""
    return turning(bases).commutator(directions)


# The origin's own base point.
origin_base = base_points(origin)                                               # [] Point-


def lifts(bases: Point) -> Point:
    """A point on the circle above each base point, `Point-`: the origin moved across the circles along the step that
    the direction from its own base point toward each is the image of, by half the angle between them."""
    cosines = polar(origin_base) & bases                                        # Scalar+
    directions = (bases - origin_base * cosines).normalized()                   # Point-
    lifting = (moving(crossing(origin_base, directions)) * (-cosines.arccos() / 4)).exp()   # Motor+
    return lifting >> origin


# The decagons' base points, the corners of an icosahedron.
decagons = scenarios.guardrails(cell_vertices, scenarios.n_guardrail_sides)      # [decagons, sides]
decagon_bases = base_points(cell_vertices.batch()[decagons[:, 0]])              # [decagons] Point-

## The travelling quadrupole

The waves are the 600-cell's gravitational waves, held in the squared lengths of its edges by Regge calculus; the core gives their stiffness, the inertia of space and the projection onto the waves, `core.gravitational_waves`, steps them by leapfrog, and blends them into a smooth field of strain over the tetrahedra and over time, `core.Waves`. The quadrupole is two of them: the squash, which stretches space along `x` and squeezes it along `y` in the frames, and the squash turned an eighth of a turn in the plane `xy`, a quarter swing behind it. Together they turn the stretch round in the plane `xy`, and on the base the quadrupole travels round. A squash is the same everywhere in the frames, so it is the same all along every circle. The waves are kept from long before the frames to as far past them as the light runs. Stepping the waves takes about seven seconds, most of it reading their strain at every stored moment.

In [ ]:
# The swing of space, and the squash and its turned copy's lags; the steps to a swing, and the swings the gauges settle
# over before the frames; the light's steps, their length and count, a quarter turn.
swing = 0.085
lags = np.array([0, np.pi / 2])                                                 # [patterns]
n_steps_per_swing = 40
n_settling_swings = 8
light_step = 0.02
n_light_steps = int(np.pi / 2 / light_step)

In [ ]:
stiffness, masses, onto_waves = core.gravitational_waves(cell_mesh, cell_vertices)


def round_steps(graph: Topology, places: Point) -> Point:
    """Each edge's round step, `Point-[edges]`, a step of the origin: its head carried back by its tail's
    frame, less the origin it is beside."""
    tails, heads = graph.incidence[1, 0] * places                               # Point-[edges], Point-[edges]
    return (frame(tails).inverse() >> heads).cast(Step)


# The squash, and the squash turned an eighth of a turn in the plane `xy`: the planes that read the directions they
# stretch and squeeze.
eighth = (mv.xy * (-np.pi / 8)).exp()                                           # [] Motor+
stretched = stack((mv.x, eighth >> mv.x))                                       # [patterns] Plane+
squeezed = stack((mv.y, eighth >> mv.y))                                        # [patterns] Plane+
steps = round_steps(cell_mesh, cell_vertices)                                   # Point-[edges]
chords = cell_mesh.d[0] * cell_vertices                                         # Point-[edges]
# Each squash's change of the squared lengths, per unit swing of space, kept to the waves: each round step's share
# along the stretched direction less its share along the squeezed one, twice, as a squared length changes twice as
# fast as the length.
shares = ((stretched & steps).squared() - (squeezed & steps).squared()) / (polar(steps) & steps)   # [patterns] Scalar+[edges]
patterns = onto_waves(2 * chords.dot(chords) * shares)                          # [patterns] Scalar+[edges]
squash = patterns[0]                                                            # Scalar+[edges]
squash_squared = float(((stiffness(squash) & squash).sites.sum() / (masses(squash) & squash).sites.sum()).to_array())
print(f"the squash's frequency squared: {squash_squared:.3f}")

In [ ]:
frequency = np.sqrt(squash_squared)
period = 2 * np.pi / frequency
level_step = period / n_steps_per_swing
# From the settling swings before the frames to the end of their swing, and as far again as the light runs.
first_time = -(n_settling_swings + 1) * period
n_levels = int(np.ceil((period + n_light_steps * light_step - first_time) / level_step)) + 2
# The squash and its turned copy, as cosines at the frequency.
phases = frequency * first_time - lags                                          # [patterns]
strains = swing * (np.cos(phases) * patterns).sum(axis=0)                       # Scalar+[edges]
rates = -swing * frequency * (np.sin(phases) * patterns).sum(axis=0)            # Scalar+[edges]
leapfrog = core.waves_stepped(stiffness, masses, onto_waves, strains, rates, level_step)
levels = [state for _, state in zip(range(n_levels), leapfrog)]
waves = core.Waves(cell_mesh, cell_vertices, [changes for changes, _ in levels],
                   [change_rates for _, change_rates in levels], first_time, level_step)

## The stretch along the circles

Read along a circle, the strain against the round metric is how much the circle stretches. It is averaged here over a few points round each circle, which fall between the tenth turns that carry the 600-cell onto itself, so that the mesh's own pattern does not decide where it is read. The base is drawn as an icosphere, its points the steps of the origin below the circles read.

In [ ]:
# The points sampled round each circle.
n_circle_samples = 7

In [ ]:
sphere_mesh, bases = scenarios.base_sphere()
pick = core.picking(cell_mesh, cell_vertices)


def stretches(bases: Point, directions: Point, time: float) -> Scalar:
    """How much lengths stretch along the steps of the origin `directions` in the frames, at the circles over the base
    points, averaged round each, at a time, `Scalar+[bases]`."""
    shares = np.arange(n_circle_samples) * scenarios.n_guardrail_sides / n_circle_samples   # [samples]
    turns = (circling * shares[:, None]).exp()                                 # [samples] Motor+
    places = turns >> lifts(bases)[None]                                        # [samples, bases] Point-
    strain, _, _, _ = waves.fields(pick(places), places, time)                 # [samples, bases] Plane+ <- Point-
    # The strain read against the metric it changes.
    readings = (strain(directions) & directions) / (2 * (polar(directions) & directions))   # [samples, bases] Scalar+
    return readings.sum(axis=0) / n_circle_samples

## Triangle gauges

In each face of the icosahedron the decagons' base points make, a triangle of atoms stands with its corners drawn toward the face's middle. Lifted from the circle over the middle across the circles, its bonds run across them too, all but a little: the circles twist about the triangle's area, so no closed triangle runs across them everywhere. The atoms roll through the waves on springs along the bonds, tuned so that the triangle's shear swings at the waves' frequency and damped, and each atom is held to its place, a free particle at rest, by a soft spring damped critically. Without that hold a triangle near the poles of the turning, where the quadrupole turns like a spinning stretch, takes up the wave's angular momentum and spins.

In [ ]:
# The corners' share of the way from a face's middle to its corners; the hold's frequency as a share of the waves';
# the shear swing's damping as a share of critical. The frames over one swing, and the time step of the atoms.
gauge_share = 0.4
holding_share = 1 / 4
gauge_damping = 0.4
n_frames = 75
atom_step = 0.01

In [ ]:
# The icosahedron's faces, and in each a triangle with its corners drawn toward the face's middle.
icosahedron = scenarios.base_hull(decagon_bases)
corners = icosahedron.incidence[2, 0] * decagon_bases.field()                   # [corners] Point-[faces]
middles = corners.sum(axis=0).normalized()                                      # Point-[faces]
atom_bases = (middles + gauge_share * (corners - middles)).normalized()         # [corners] Point-[faces]
# Each atom's direction along the base from its face's middle and the cosine of the angle between them: the step
# across the circles that the direction is the image of carries the middle's lift half that angle.
cosines = polar(middles) & atom_bases                                           # [corners] Scalar+[faces]
directions = (atom_bases - middles * cosines).normalized()                      # [corners] Point-[faces]
moves = (moving(crossing(middles, directions)) * (-cosines.arccos() / 4)).exp()   # [corners] Motor+[faces]
face_atoms = (frame(lifts(middles)) * moves) >> origin                          # [corners] Point-[faces]
# The atoms three to a face, face after face, and each triangle's three sides.
atoms = stack(tuple(face_atoms.batch()), axis=-1).reshape(-1).field()           # Point-[atoms]
bonds = Topology.loops([3] * icosahedron.n_elements[2])

In [ ]:
# The hold, and the bonds' springs and dashpots: a triangle of unit masses swings in shear at the root of three halves
# of its bonds' stiffness and the hold, damped by three halves of its bonds' dashpots and by the hold's.
holding = (holding_share * frequency) ** 2
n_bonds = bonds.n_elements[1]
bond_stiffness = mv.scalar(np.full((n_bonds, 1), 2 / 3 * (frequency ** 2 - holding))).field()   # Scalar+[bonds]
bond_damping = mv.scalar(np.full((n_bonds, 1), 2 / 3 * (2 * gauge_damping * frequency - 2 * np.sqrt(holding)))).field()   # Scalar+[bonds]
bond_steps = round_steps(bonds, atoms)                                          # Point-[bonds]
bond_rest = bond_steps.dot(bond_steps).square_root()                            # Scalar+[bonds]
times = np.arange(n_frames) * period / n_frames                                 # [frames]
at_rest = mv(Step, np.zeros((bonds.n_vertices, 3))).field()                     # Point-[atoms]
states = core.massive(pick, waves, bonds, bond_rest, bond_stiffness, bond_damping, holding, atoms, at_rest,
                      times[0] - n_settling_swings * period, atom_step, times)

## A bundle of light

A narrow bundle of rays sets out across the circles from the circle over the base point along `z`, its headings the base's directions about the one toward `y`, which faces the viewer, fanned in the plane `xy` square to the source and each turned a quarter in the plane square to the base point. Through the round sphere every ray would run along a great circle across the circles and reach the base's far pole after a quarter turn; through the waves the rays sway, and their paths are seen on the base. A fresh bundle sets out at every frame.

In [ ]:
# The rays, and the angle they are spread over.
n_rays = 7
light_spread = np.radians(18)

In [ ]:
source, target = mv.z.dual(), mv.y.dual()                                       # [] Point-, [] Point-
aim = (target - source * (polar(source) & target)).normalized()                 # [] Point-
fan = (mv.xy * (-np.linspace(-light_spread / 2, light_spread / 2, n_rays) / 2)).exp()   # [rays] Motor+
headings = crossing(source, fan >> aim).normalized()                            # [rays] Point-
starts = lifts(source[None]).broadcast_to((n_rays,))                            # [rays] Point-

## The base from outside

Each frame shows the base from outside at one moment of the swing: the circles' stretch, the gauges and the light, all read on the base. The last cell takes about half a minute, most of it rolling the atoms, tracing the light and drawing the frames.

In [ ]:
# The points drawn along each bond, and a frame's duration in milliseconds.
n_arc_points = 12
duration_ms = 40

In [ ]:
def on_base() -> Iterator[tuple]:
    """At each frame, all on the base: the circles' stretch at the icosphere's points, the atoms, the bonds' arcs and
    strains, and the light's paths."""
    shares = np.linspace(0, 1, n_arc_points)                                    # [arc points]
    for time, (positions, strains) in zip(times, states):
        tails, heads = bonds.incidence[1, 0] * positions                       # Point-[bonds], Point-[bonds]
        arc_points = (tails.batch()[:, None] * (1 - shares) + heads.batch()[:, None] * shares).normalized()   # [bonds, arc points] Point-
        light_places = core.light(pick, waves, starts, headings, time, light_step, n_light_steps)
        paths = stack([starts, *light_places], axis=-1)                         # [rays, steps + 1] Point-
        yield (stretches(bases, bases, time), base_points(positions.batch()), base_points(arc_points),
               strains.batch(), base_points(paths))


show_animation((render.base_view(sphere_mesh, bases, decagon_bases, *state) for state in on_base()),
               "closed_universe_quadrupole", duration_ms)

The base is shaded by how much the circles above it stretch, red where they lengthen and blue where they shorten. The decagons' base points are dots in their own colours; the triangles' bonds are drawn as the arcs they make on the base, coloured by their strain; and the light's paths are the dark lines.

In [ ]:
# checks
# The quadrupole is the same all along each circle: a circle's stretch read at one point of it is its average round
# it; the triangles' atoms stand over their corners; and the light sets out across the circles.
places = lifts(bases)                                                           # [points] Point-
strain, _, _, _ = waves.fields(pick(places), places, 0.0)                       # [points] Plane+ <- Point-
at_one = (strain(bases) & bases) / (2 * (polar(bases) & bases))                 # [points] Scalar+
averaged = stretches(bases, bases, 0.0)                                         # [points] Scalar+
assert ((at_one - averaged).abs() < 1e-6 * np.abs(averaged.to_array()).max()).all()
bases_of_atoms = stack(tuple(atom_bases.batch()), axis=-1).reshape(-1)          # [atoms] Point-
assert (((polar(base_points(atoms.batch())) & bases_of_atoms) - 1).abs() < 1e-12).all()
assert ((polar(base_points(starts)) & headings).abs() < 1e-12).all()